In [1]:
import gymnasium as gym
import torch

# from pystk2_gymnasium import AgentSpec
from dagger_rlpd_stk import (
    DDPGConfig,
    RLPDConfig,
    SACConfig,
    create_offline_replay_buffer,
    run_ddpg,
    run_rlpd,
    run_sac,
)

## DDPG

In [ ]:
# ddpg_config = DDPGConfig(
#     start_timesteps=25_000,
#     max_timesteps=500_000,
#     batch_size=256,
#     eval_freq=5_000,
#     eval_episodes=10,
#     exploration_noise=0.1,
#     seed=0,
# )

In [ ]:
# env = gym.make("LunarLander-v3", continuous=True)
# evaluation_env = gym.make("LunarLander-v3", continuous=True)

# ddpg_policy, ddpg_evaluations = run_ddpg(
#     env,
#     evaluation_env,
#     ddpg_config,
#     log_dir="runs/lunar_lander_ddpg",
# )

# env.close()
# evaluation_env.close()

## SAC

In [ ]:
sac_config = SACConfig(
    start_timesteps=25_000,  # 25_000,
    max_timesteps=500_000,
    batch_size=256,
    eval_freq=5_000,
    eval_episodes=10,
    initial_alpha=0.2,
    seed=0,
)

In [ ]:
# env_name = "supertuxkart/flattened_continuous_actions-v0"
# player_name = "Jean"

# env = gym.make(
#     env_name,
#     render_mode=None,
#     agent=AgentSpec(name=player_name),
# )
# evaluation_env = gym.make(
#     env_name,
#     render_mode=None,
#     agent=AgentSpec(name=player_name),
# )

env = gym.make("LunarLander-v3", continuous=True)
evaluation_env = gym.make("LunarLander-v3", continuous=True)


sac_policy, sac_evaluations = run_sac(
    env,
    evaluation_env,
    sac_config,
    log_dir="runs/lunar_lander_sac",
)

env.close()
evaluation_env.close()

In [ ]:
torch.save(sac_policy, "LunarLanderSacPolicy.pth")

In [ ]:
sac_policy = torch.load("LunarLanderSacPolicy.pth", weights_only=False)

In [ ]:
rlpd_config = RLPDConfig(
    max_timesteps=100_000,
    n_ensemble_critics=3,
    n_subset_critics=2,
    utd_ratio=5,
    batch_size=64,
)

In [ ]:
offline_env = gym.make("LunarLander-v3", continuous=True)
offline_buffer = create_offline_replay_buffer(
    sac_policy, offline_env, rlpd_config.replay_buffer_size
)
offline_env.close()

In [ ]:
env = gym.make("LunarLander-v3", continuous=True)
evaluation_env = gym.make("LunarLander-v3", continuous=True)

rlpd_policy, rlpd_evaluations = run_rlpd(
    env,
    evaluation_env,
    offline_buffer,
    rlpd_config,
    log_dir="runs/lunar_lander_rlpd",
)